In [1]:
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms

def seed_everything(seed=42):
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [2]:
import os, glob, tarfile

src = '/kaggle/input/datasets/amirsafarzadeh8203/cifar-10-python-tar-gz'

found = glob.glob(src + '/**/cifar-10-batches-py', recursive=True)
if found:
    root = os.path.dirname(found[0])
else:
    tar_path = glob.glob(src + '/**/*.tar*', recursive=True)[0]
    os.makedirs('/kaggle/working/data', exist_ok=True)
    with tarfile.open(tar_path) as tar:
        tar.extractall('/kaggle/working/data')
    root = '/kaggle/working/data'

print(root)
print(os.listdir(os.path.join(root, 'cifar-10-batches-py')))

/kaggle/input/datasets/amirsafarzadeh8203/cifar-10-python-tar-gz
['data_batch_1', 'data_batch_2', 'batches.meta', 'test_batch', 'data_batch_3', 'data_batch_5', 'data_batch_4', 'readme.html']


In [3]:
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

seed_everything(42)

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(224, padding=28),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_full = datasets.CIFAR10(root, train=True, download=False, transform=train_transform)
val_full   = datasets.CIFAR10(root, train=True, download=False, transform=test_transform)
test_dataset = datasets.CIFAR10(root, train=False, download=False, transform=test_transform)

train_idx, val_idx = train_test_split(range(len(train_full)), test_size=0.1, random_state=42,
                                      stratify=train_full.targets)

train_subset = Subset(train_full, train_idx)  # with augmentation
val_subset   = Subset(val_full, val_idx)      # without augmentation

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)

val_loader   = DataLoader(val_subset, batch_size=64, shuffle=False,
                          num_workers=4, pin_memory=True, persistent_workers=True)

test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False,
                          num_workers=4, pin_memory=True, persistent_workers=True)


print(f"Train size: {len(train_subset)}, Val size: {len(val_subset)}, Test size: {len(test_dataset)}")
print(f"train classes: {train_full.classes}")

Train size: 45000, Val size: 5000, Test size: 10000
train classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [4]:
def build_model(dropout_p=0.3):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    for param in model.parameters():
        param.requires_grad = False
    for param in model.layer4.parameters():
        param.requires_grad = True

    model.fc = nn.Sequential(
        nn.Dropout(p=dropout_p),
        nn.Linear(model.fc.in_features, 10),
    )
    return model.to(device)

In [5]:
def evaluate_loss(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
    return total_loss / len(loader)

In [6]:
import optuna

def objective(trial):
    seed_everything(42)

    layer4_lr = trial.suggest_float('layer4_lr', 1e-5, 1e-3, log=True)
    fc_lr = trial.suggest_float('fc_lr', 1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-1, log=True)
    dropout_p = trial.suggest_float('dropout_p', 0.1, 0.5)

    model = build_model(dropout_p=dropout_p)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam([
        {'params': model.layer4.parameters(), 'lr': layer4_lr},
        {'params': model.fc.parameters(), 'lr': fc_lr},
    ], weight_decay=weight_decay)
    scaler = torch.amp.GradScaler(device)

    for epoch in range(1,5):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast(device_type=device.type, dtype=torch.float16):
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        val_loss = evaluate_loss(model, val_loader, criterion, device)
        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return val_loss

In [7]:
study = optuna.create_study(direction='minimize', pruner=optuna.pruners.MedianPruner(n_warmup_steps=1))
study.optimize(objective, n_trials=15)

print('Best params:', study.best_params)
print('Best val_loss:', study.best_value)

[I 2026-09-22 22:02:44,669] A new study created in memory with name: no-name-3824a6fd-2066-4408-a497-0d03af595719


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 226MB/s]
[I 2026-09-22 22:07:22,407] Trial 0 finished with value: 0.2353083924685098 and parameters: {'layer4_lr': 1.4774346661277114e-05, 'fc_lr': 0.00036879643796355335, 'weight_decay': 0.0025450732948917223, 'dropout_p': 0.29908679783961734}. Best is trial 0 with value: 0.2353083924685098.
[I 2026-09-22 22:11:52,801] Trial 1 finished with value: 0.2305901034534732 and parameters: {'layer4_lr': 0.00027158537073836874, 'fc_lr': 0.0002744864733081889, 'weight_decay': 0.0004001991278900568, 'dropout_p': 0.1274371543263178}. Best is trial 1 with value: 0.2305901034534732.
[I 2026-09-22 22:16:22,969] Trial 2 finished with value: 0.5685057243968867 and parameters: {'layer4_lr': 0.0008788071905355853, 'fc_lr': 0.001973194515152937, 'weight_decay': 0.035132291428797126, 'dropout_p': 0.22416581586811565}. Best is trial 1 with value: 0.2305901034534732.
[I 2026-09-22 22:20:54,384] Trial 3 finished with value: 0.20075994748857956 and parameters: {'laye

Best params: {'layer4_lr': 8.957680031001757e-05, 'fc_lr': 0.00010007378780798995, 'weight_decay': 6.819429058550563e-05, 'dropout_p': 0.4899448573056766}
Best val_loss: 0.19671077238796633
